# <center><span style="color:#336699">Introdução à Ciência de Dados</span></center>
<hr style="border:2px solid #0077b9;">

<br/>

<div style="text-align: center;font-size: 150%;"> 
    Análise das Citações do GEOINFO</br>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
    Gabrielly Prado e Giulia Tomazeli
    <br/><br/>
    Mestrado em Computação Aplicada, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Última Atualização: 25 de Agosto de 2026
</div>

<br/>

<div style="text-align: center;"> 
    <b>Resumo.</b> Este Jupyter Notebook apresenta o pré-processamento dos dados da base bibliométrica dos trabalhos que citam as publicações do GEOINFO.
</div>

# Bibliotecas

In [ ]:
import sys
sys.path.append("..") 

import pandas as pd

from config.configs import (
    PATH_GEOINFO,
    PATH_CITACOES_ENRIQUECIDAS,
    PATH_GOOGLE_SCHOLAR,
    PATH_OPENALEX,
    OUTPUT_DIR,
    COLUNAS_PADRAO,
    COLUNAS_TEXTO,
)

from src.utils.preprocessing import (
    limpar_artigos,
    normalizar_colunas_textuais,
    padronizar_ano,
    padronizar_numero_edicao,
    padronizar_doi,
    padronizar_openalex_id,
    limpar_url,
    padronizar_status_openalex,
    padronizar_paises,
    padronizar_idioma,
    dividir_instituicoes_numeradas,
    reparar_autores_nao_divididos,
    garantir_colunas,
    verificar_estrutura,
    resumo_nulos,
    resumo_tipos,
    padronizar_autores,
    padronizar_instituicoes,
    gerar_tabela_valores_unicos,
    padronizar_autor,
    gerar_mapa_canonico_automatico
)

# Carregar dados

In [26]:
df_geoinfo = pd.read_csv(PATH_GEOINFO)
df_citacoes_enriquecidas = pd.read_csv(PATH_CITACOES_ENRIQUECIDAS)
df_scholar = pd.read_csv(PATH_GOOGLE_SCHOLAR)
df_openalex = pd.read_csv(PATH_OPENALEX)

for nome, df in [
    ("GEOINFO", df_geoinfo),
    ("Citações enriquecidas", df_citacoes_enriquecidas),
    ("Google Scholar", df_scholar),
    ("OpenAlex", df_openalex),
]:
    print(f"{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")
    print(list(df.columns))
    print()

GEOINFO: 681 linhas, 11 colunas
['titulo', 'ano', 'autores', 'instituicoes', 'edicao', 'identificador', 'idioma', 'url_edicao', 'url_artigo', 'url_metadata', 'numero_edicao']

Citações enriquecidas: 2988 linhas, 20 colunas
['titulo_original', 'ano_original', 'titulo', 'ano', 'autores', 'instituicoes', 'idioma', 'pais', 'veiculo_publicacao', 'tipo_documento', 'doi', 'url', 'topico', 'subcampo', 'campo', 'dominio_tematico', 'fonte_publicacao', 'status_acesso_aberto', 'openalex_id', 'openalex_status']

Google Scholar: 2988 linhas, 12 colunas
['titulo_original', 'ano_original', 'titulo', 'ano', 'autores', 'instituicoes', 'idioma', 'pais', 'veiculo_publicacao', 'tipo_documento', 'doi', 'url']

OpenAlex: 0 linhas, 19 colunas
['titulo_original', 'ano_original', 'url_original', 'titulo', 'ano', 'autores', 'instituicoes', 'pais', 'idioma', 'doi', 'tipo_documento', 'topico', 'subcampo', 'campo', 'dominio_tematico', 'fonte_publicacao', 'status_acesso_aberto', 'openalex_id', 'url']



# Limpeza geral

In [27]:
df_geoinfo = limpar_artigos(df_geoinfo)
df_citacoes_enriquecidas = limpar_artigos(df_citacoes_enriquecidas)
df_scholar = limpar_artigos(df_scholar)
df_openalex = limpar_artigos(df_openalex)

In [28]:
df_geoinfo["instituicoes"] = df_geoinfo["instituicoes"].apply(dividir_instituicoes_numeradas)
df_citacoes_enriquecidas["autores"] = df_citacoes_enriquecidas["autores"].apply(reparar_autores_nao_divididos)

# Normalização de texto e encoding

In [ ]:
df_geoinfo = normalizar_colunas_textuais(df_geoinfo, COLUNAS_TEXTO, corrigir_geoinfo=True)

df_citacoes_enriquecidas = normalizar_colunas_textuais(df_citacoes_enriquecidas, COLUNAS_TEXTO, corrigir_geoinfo=False)
df_scholar = normalizar_colunas_textuais(df_scholar, COLUNAS_TEXTO, corrigir_geoinfo=False)
df_openalex = normalizar_colunas_textuais(df_openalex, COLUNAS_TEXTO, corrigir_geoinfo=False)

# Padronizar campos 

In [30]:
def padronizar_campos_comuns(df):
    df = df.copy()

    if "ano" in df.columns:
        df["ano"] = df["ano"].apply(padronizar_ano)
    if "ano_original" in df.columns:
        df["ano_original"] = df["ano_original"].apply(padronizar_ano)

    if "numero_edicao" in df.columns:
        df["numero_edicao"] = df["numero_edicao"].apply(padronizar_numero_edicao)

    if "doi" in df.columns:
        df["doi"] = df["doi"].apply(padronizar_doi)

    if "openalex_id" in df.columns:
        df["openalex_id"] = df["openalex_id"].apply(padronizar_openalex_id)

    if "openalex_status" in df.columns:
        df["openalex_status"] = df["openalex_status"].apply(padronizar_status_openalex)

    if "pais" in df.columns:
        df["pais"] = df["pais"].apply(padronizar_paises)

    if "idioma" in df.columns:
        df["idioma"] = df["idioma"].apply(padronizar_idioma)

    for coluna_url in ["url", "url_original", "url_artigo", "url_metadata", "url_edicao"]:
        if coluna_url in df.columns:
            df[coluna_url] = df[coluna_url].apply(limpar_url)

    return df


df_geoinfo = padronizar_campos_comuns(df_geoinfo)
df_citacoes_enriquecidas = padronizar_campos_comuns(df_citacoes_enriquecidas)
df_scholar = padronizar_campos_comuns(df_scholar)
df_openalex = padronizar_campos_comuns(df_openalex)

# Marcar origem

In [ ]:
df_geoinfo["fonte_dados"] = "geoinfo"
df_citacoes_enriquecidas["fonte_dados"] = "citacoes_enriquecidas"
df_openalex["fonte_dados"] = "openalex"

# Garantir estrutura igual

In [32]:
df_geoinfo_padrao = garantir_colunas(df_geoinfo, COLUNAS_PADRAO)
df_citacoes_padrao = garantir_colunas(df_citacoes_enriquecidas, COLUNAS_PADRAO)
df_openalex_padrao = garantir_colunas(df_openalex, COLUNAS_PADRAO)

In [33]:
verificar_estrutura({
    "GEOINFO": df_geoinfo_padrao,
    "Citações enriquecidas": df_citacoes_padrao,
    "OpenAlex": df_openalex_padrao,
})

[OK] Estruturas padronizadas.


True

# Identificador único

In [ ]:
df_citacoes = df_citacoes_padrao.copy()
print(f"Total de citações: {len(df_citacoes)} linhas")

Total de citações: 2988 linhas


In [ ]:
df_citacoes["identificador"] = ("citacao_" + df_citacoes.index.astype(str))

# Nulos e tipos

In [36]:
resumo_nulos(df_geoinfo_padrao, "GEOINFO (artigos originais)")
resumo_tipos(df_geoinfo_padrao, "GEOINFO (artigos originais)")

resumo_nulos(df_openalex_padrao, "OpenAlex (artigos citantes)")
resumo_tipos(df_openalex_padrao, "OpenAlex (artigos citantes)")

resumo_nulos(df_citacoes, "Citações (base enriquecida final)")
resumo_tipos(df_citacoes, "Citações (base enriquecida final)")


GEOINFO (artigos originais)
                      nulos  percentual
titulo_original         681       100.0
ano_original            681       100.0
tipo_documento          681       100.0
doi                     681       100.0
veiculo_publicacao      681       100.0
pais                    681       100.0
status_acesso_aberto    681       100.0
fonte_publicacao        681       100.0
dominio_tematico        681       100.0
campo                   681       100.0
subcampo                681       100.0
topico                  681       100.0
url                     681       100.0
url_original            681       100.0
openalex_id             681       100.0
openalex_status         681       100.0
identificador             0         0.0
titulo                    0         0.0
instituicoes              0         0.0
idioma                    0         0.0
ano                       0         0.0
autores                   0         0.0
url_artigo                0         0.0
url_metadat

# Instituições e autores

In [ ]:
for df in [df_geoinfo_padrao, df_citacoes]:
    df["instituicoes"] = df["instituicoes"].apply(padronizar_instituicoes)
    df["autores"] = df["autores"].apply(padronizar_autor)      # normalização automática 
    df["autores"] = df["autores"].apply(padronizar_autores)    # depois o mapa manual, para casos residuais

In [38]:
df_geoinfo_padrao["autores"] = df_geoinfo_padrao["autores"].apply(dividir_instituicoes_numeradas)
df_citacoes["autores"] = df_citacoes["autores"].apply(dividir_instituicoes_numeradas)

In [39]:
instituicoes = pd.concat([
    gerar_tabela_valores_unicos(df_geoinfo_padrao, "instituicoes", "geoinfo"),
    gerar_tabela_valores_unicos(df_citacoes, "instituicoes", "citantes"),
], ignore_index=True)

autores = pd.concat([
    gerar_tabela_valores_unicos(df_geoinfo_padrao, "autores", "geoinfo"),
    gerar_tabela_valores_unicos(df_citacoes, "autores", "citantes"),
], ignore_index=True)

print(f"Instituições com possível duplicata: {instituicoes['possivel_duplicata'].sum()}")
print(f"Autores com possível duplicata: {autores['possivel_duplicata'].sum()}")

Instituições com possível duplicata: 0
Autores com possível duplicata: 154


In [40]:
for nome_norm, grupo in instituicoes[instituicoes["possivel_duplicata"]].groupby("nome_normalizado"):
    print(f"\n--- grupo: {nome_norm!r} ---")
    for _, linha in grupo.iterrows():
        print(f"  {linha['ocorrencias']:>3}x  [{linha['origem']}]  {linha['instituicoes']!r}")

In [41]:
for nome_norm, grupo in autores[autores["possivel_duplicata"]].groupby("nome_normalizado"):
    print(f"\n--- grupo: {nome_norm!r} ---")
    for _, linha in grupo.iterrows():
        print(f"  {linha['ocorrencias']:>3}x  [{linha['origem']}]  {linha['autores']!r}")


--- grupo: 'almeida claudia maria de' ---
    2x  [geoinfo]  'Almeida, Cláudia Maria de'
    1x  [geoinfo]  'Almeida, Claudia Maria de'

--- grupo: 'almeida claudio aparecido de' ---
    1x  [geoinfo]  'Almeida, Claudio Aparecido de'
    1x  [geoinfo]  'Almeida, Cláudio Aparecido de'

--- grupo: 'almeida junior jose r' ---
    1x  [geoinfo]  'Almeida Júnior, José R.'
    1x  [geoinfo]  'Almeida Junior, José R.'

--- grupo: 'anderson liana o' ---
    2x  [geoinfo]  'Anderson, Liana O.'
    1x  [geoinfo]  'Anderson, Liana O'

--- grupo: 'andrade fabio gomes de' ---
    3x  [geoinfo]  'Andrade, Fabio Gomes de'
    1x  [geoinfo]  'Andrade, Fábio Gomes de'

--- grupo: 'andrade marcus v a' ---
    8x  [geoinfo]  'Andrade, Marcus V. A.'
    4x  [geoinfo]  'Andrade, Marcus V. A'

--- grupo: 'andrade pedro r' ---
    1x  [geoinfo]  'Andrade, Pedro R.'
    1x  [geoinfo]  'Andrade, Pedro R'

--- grupo: 'anjos abner ernani dos' ---
    1x  [geoinfo]  'Anjos, Abner Ernani dos'
    1x  [geoinfo]  '

In [42]:
mapa_autores_auto = gerar_mapa_canonico_automatico(autores, "autores")
mapa_instituicoes_auto = gerar_mapa_canonico_automatico(instituicoes, "instituicoes")

print(f"Pares de autores a corrigir automaticamente: {len(mapa_autores_auto)}")
print(f"Pares de instituições a corrigir automaticamente: {len(mapa_instituicoes_auto)}")

Pares de autores a corrigir automaticamente: 154
Pares de instituições a corrigir automaticamente: 4


In [ ]:
def aplicar_mapa_canonico(valor, mapa, separador=", "):
    if pd.isna(valor):
        return valor
    partes = [p.strip() for p in str(valor).split(separador) if p.strip()]
    partes_corrigidas = [mapa.get(p, p) for p in partes]
    partes_unicas = list(dict.fromkeys(partes_corrigidas))
    return separador.join(partes_unicas)


df_geoinfo_padrao["autores"] = df_geoinfo_padrao["autores"].apply(
    lambda v: aplicar_mapa_canonico(v, mapa_autores_auto, separador="; ")
)

df_citacoes["autores"] = df_citacoes["autores"].apply(
    lambda v: aplicar_mapa_canonico(v, mapa_autores_auto, separador="; ")
)

In [45]:
instituicoes = pd.concat([
    gerar_tabela_valores_unicos(df_geoinfo_padrao, "instituicoes", "geoinfo"),
    gerar_tabela_valores_unicos(df_citacoes, "instituicoes", "citantes"),
], ignore_index=True)

autores = pd.concat([
    gerar_tabela_valores_unicos(df_geoinfo_padrao, "autores", "geoinfo"),
    gerar_tabela_valores_unicos(df_citacoes, "autores", "citantes"),
], ignore_index=True)

print(f"Instituições com possível duplicata: {instituicoes['possivel_duplicata'].sum()}")
print(f"Autores com possível duplicata: {autores['possivel_duplicata'].sum()}")

Instituições com possível duplicata: 0
Autores com possível duplicata: 0


# Salvar

In [46]:
caminho_geoinfo = OUTPUT_DIR / "geoinfo_final.csv"
caminho_citacoes = OUTPUT_DIR / "citacoes_final.csv"

df_geoinfo_padrao.to_csv(caminho_geoinfo, index=False, encoding="utf-8-sig")
df_citacoes.to_csv(caminho_citacoes, index=False, encoding="utf-8-sig")

print(f"Salvo: {caminho_geoinfo}")
print(f"Salvo: {caminho_citacoes}")

Salvo: ..\data\processed\geoinfo_final.csv
Salvo: ..\data\processed\citacoes_final.csv


In [47]:
instituicoes.to_csv(OUTPUT_DIR / "instituicoes_padronizadas.csv", index=False, encoding="utf-8-sig")
autores.to_csv(OUTPUT_DIR / "autores_padronizados.csv", index=False, encoding="utf-8-sig")